# Train one member (Colab driver)

Runs `python -m rogii_wellbore.run` for one member config. Every finished (fold, seed) job is
written to Drive as it completes, and a re-run skips finished jobs, so a dead session can be
resumed by running the notebook again.

Set `MEMBER` to one of `configs/members/*.yaml`. With `PRETRAINED_DIR` pointing at a folder of
`pretrain_f{fold}_s{seed}_ep5.pt` files (how the final members were trained) the synthetic
pretraining is skipped; with `None` each job builds its fold's 25,000-well pool and pretrains.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MEMBER         = 'v2l'                                   # v2l | v2xl | v2l_small | l2
DRIVE_ROOT     = '/content/drive/MyDrive/Kaggle/ROGII-Wellbore'
DATA_ZIP       = f'{DRIVE_ROOT}/rogii-wellbore-geology-prediction.zip'
PRETRAINED_DIR = None                                    # or a run folder holding pretrain_f*_s*_ep5.pt
FOLDS, SEEDS   = None, None                              # None = the config's; e.g. [0], [34]

In [ ]:
import os, subprocess
if not os.path.exists('/content/rogii-wellbore-2026'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/TuckerArrants/rogii-wellbore-2026.git',
                    '/content/rogii-wellbore-2026'], check=True)
subprocess.run(['pip', 'install', '-q', '-e', '/content/rogii-wellbore-2026'], check=True)
if not os.path.exists('/content/rogii/train'):
    subprocess.run(['unzip', '-q', DATA_ZIP, '-d', '/content/rogii'], check=True)
os.environ['HF_HOME'] = f'{DRIVE_ROOT}/hf_home'            # encoder weights cached on Drive, downloaded once

In [ ]:
from rogii_wellbore import config as C
from rogii_wellbore.run import run

cfg = C.load(f'/content/rogii-wellbore-2026/configs/members/{MEMBER}.yaml')
out = os.path.join(DRIVE_ROOT, cfg.experiment_name())
print(cfg.name, '->', out)
summary = run(cfg, '/content/rogii', out, pretrained_dir=PRETRAINED_DIR, folds_subset=FOLDS, seeds_subset=SEEDS)
print(summary['full_cv'])